# **2. DATA UNDERSTANDING**

## **LIBRARIES**

In [16]:
! pip install adlfs azure-storage-blob

In [17]:
# Standard Library
import os
import io
from io import BytesIO

# DATA MANIPULATION AND ANALYSIS
import pandas as pd
import numpy as np

# AZURE CLOUD STORAGE
from adlfs import AzureBlobFileSystem
from azure.storage.blob import BlobServiceClient

# Regular expressions
import re

## **EXTRACT FROM AZURE BLOB STORAGE**

In [18]:
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

In [19]:
# Connection configuration
AZURE_STORAGE_ACCOUNT = "researchprojectx24104515"
AZURE_STORAGE_KEY = "bxpexO6i+Hz6n1WiipTn+sTCuLPGMS1BogMERrIrHd16DpQ0GLfQ0R33yrSw4MxsDomq5yNMgw1o+AStlx/MjA=="
CONTAINER_NAME = "footfall"

# Establish connection
connection_string = f"DefaultEndpointsProtocol=https;AccountName={AZURE_STORAGE_ACCOUNT};AccountKey={AZURE_STORAGE_KEY};EndpointSuffix=core.windows.net"
blob_service_client = BlobServiceClient.from_connection_string(connection_string)
container_client = blob_service_client.get_container_client(CONTAINER_NAME)

fs = AzureBlobFileSystem(
    account_name=AZURE_STORAGE_ACCOUNT,
    account_key=AZURE_STORAGE_KEY
)

### **FROM BLOB TO DATAFRAME**

In [20]:
def load_csv_from_blob(blob_name):
    blob_client = container_client.get_blob_client(blob_name)
    stream = blob_client.download_blob()
    data_bytes = stream.readall()
    data_str = data_bytes.decode("utf-8")
    return pd.read_csv(io.StringIO(data_str))

# Connection configuration
AZURE_STORAGE_ACCOUNT = "researchprojectx24104515"
AZURE_STORAGE_KEY = "bxpexO6i+Hz6n1WiipTn+sTCuLPGMS1BogMERrIrHd16DpQ0GLfQ0R33yrSw4MxsDomq5yNMgw1o+AStlx/MjA=="
connection_string = f"DefaultEndpointsProtocol=https;AccountName={AZURE_STORAGE_ACCOUNT};AccountKey={AZURE_STORAGE_KEY};EndpointSuffix=core.windows.net"
blob_service_client = BlobServiceClient.from_connection_string(connection_string)

fs = AzureBlobFileSystem(
    account_name=AZURE_STORAGE_ACCOUNT,
    account_key=AZURE_STORAGE_KEY
)

# Establish connection to counters dataset
CONTAINER_NAME = "footfall2019"
container_client = blob_service_client.get_container_client(CONTAINER_NAME)

file_name = "pedestrian-2019.csv"
p2019 = load_csv_from_blob(file_name)
print(f"Loaded '{file_name}' with shape: {p2019.shape}")

Loaded 'pedestrian-2019.csv' with shape: (35036, 24)


In [21]:
p2019.head()

,Time,O'Connell St Outside Pennys,O'Connell St Outside Clerys,Mary Street,Capel Street,Aston Quay,Grafton Street @ CompuB,Talbot Street North,"Doilier Street, Burgh Quay",Dawson Street Replacement,Dame Street,Talbot Street South,"O'Connell St, Parnell St @ AIB",Grafton Street / Nassau Street / Suffolk Street,"College Green, Bank Of Ireland",Henry Street,Westmoreland Street East,Dawson Street,Liffey Street,Westmoreland Street West,Grafton Street,Bachelors Walk,College Green @ Church Lane,College Green - Dame St Side
0,01-01-2019 00:00:00,244,1914.0,20,16,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,95.0,1670.0,255.0,881.0,1988.0,140.0,NaN,NaN,NaN
1,01-01-2019 00:15:00,454,NaN,24,62,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,265.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,01-01-2019 00:30:00,391,NaN,69,99,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,117.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,01-01-2019 00:45:00,415,NaN,50,61,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,120.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,01-01-2019 01:00:00,319,885.0,24,47,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,90.0,767.0,209.0,723.0,1270.0,215.0,NaN,NaN,NaN


## **HANDLING GRANULARITY DIFFERENT THAN THE REST OF THE DATASETS**

In [22]:
# Time in datetime format
p2019["Time"] = pd.to_datetime(p2019["Time"], dayfirst=True)

# Group every four columns and add the values up
# New Time column
p2019["NewTime"] = p2019.index // 4

# Addition of all rows in each group except Time
newdataframe = p2019.groupby("NewTime").agg({
    "Time": "first",  # First stamp of the group remains
    **{col: "sum" for col in p2019.columns if col not in ["Time", "NewTime"]}
})

# Set time as new index
newdataframe = newdataframe.set_index("Time")
newdataframe = newdataframe.reset_index()
newdataframe.head()

,Time,O'Connell St Outside Pennys,O'Connell St Outside Clerys,Mary Street,Capel Street,Aston Quay,Grafton Street @ CompuB,Talbot Street North,"Doilier Street, Burgh Quay",Dawson Street Replacement,Dame Street,Talbot Street South,"O'Connell St, Parnell St @ AIB",Grafton Street / Nassau Street / Suffolk Street,"College Green, Bank Of Ireland",Henry Street,Westmoreland Street East,Dawson Street,Liffey Street,Westmoreland Street West,Grafton Street,Bachelors Walk,College Green @ Church Lane,College Green - Dame St Side
0,2019-01-01 00:00:00,1504,1914.0,163,238,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,597.0,1670.0,255.0,881.0,1988.0,140.0,0.0,0.0,0.0
1,2019-01-01 01:00:00,1187,885.0,102,173,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,359.0,767.0,209.0,723.0,1270.0,215.0,0.0,0.0,0.0
2,2019-01-01 02:00:00,1233,984.0,63,121,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,317.0,642.0,193.0,1010.0,1589.0,210.0,0.0,0.0,0.0
3,2019-01-01 03:00:00,1316,935.0,59,174,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,313.0,582.0,178.0,772.0,1534.0,204.0,0.0,0.0,0.0
4,2019-01-01 04:00:00,802,390.0,46,82,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,172.0,143.0,35.0,197.0,610.0,88.0,0.0,0.0,0.0


In [23]:
def save_blob(data, filename):
    try:
        blob_name = f"{CONTAINER_NAME}/{filename}"
        csv_data = data.to_csv(index=False)
        with fs.open(blob_name, "w") as f:
            f.write(csv_data)
        print(f"Saved to {blob_name}")
    # Exceptions handling
    except Exception as e:
        print(f"Error saving data to blob storage: {str(e)}")
        return False

In [24]:
CONTAINER_NAME = "footfall"

save_blob(newdataframe, "pedestrian-2019.csv")

Saved to footfall/pedestrian-2019.csv


In [25]:
newdataframe.dtypes

,0
Time,datetime64[ns]
O'Connell St Outside Pennys,int64
O'Connell St Outside Clerys,float64
Mary Street,int64
Capel Street,int64
Aston Quay,float64
Grafton Street @ CompuB,float64
Talbot Street North,float64
"Doilier Street, Burgh Quay",float64
Dawson Street Replacement,float64
